# Notebook 7 — Courbes IDF (Intensité-Durée-Fréquence)
## Méthode Montana — Station de Kara, Togo

---

## Objectif

Ce notebook est le **livrable final** du pipeline. Il produit les **courbes IDF**
de la station de Kara utilisées pour le dimensionnement d'ouvrages hydrauliques.

### Formule de Montana

$$I(D, T) = a(T) \times D^{b(T)}$$

où $I$ = intensité (mm/h), $D$ = durée (min), $T$ = période de retour (ans).

### Améliorations appliquées dans cette version

1. **Durées standardisées** — on travaille uniquement avec les 8 durées standards
   issues du pluviographe nettoyé (n3 corrigé) : 30, 60, 90, 120, 180, 240, 360, 720 min
2. **Seuil minimum 5 années** — une durée est utilisée seulement si elle dispose
   d'au moins 5 années de maxima annuels (au lieu de 3)
3. **Vérification de monotonie** — on contrôle que I(D1) > I(D2) pour D1 < D2
   et on corrige par interpolation PCHIP si nécessaire
4. **Diagnostic complet** — rapport de monotonie avant/après correction

### Méthodologie

1. Maxima annuels par durée standard
2. Ajustement Gumbel par durée → valeurs de retour
3. Vérification de monotonie + correction PCHIP
4. Ajustement Montana par période de retour
5. Tableau IDF et courbes finales

### Livrables

| Fichier | Contenu |
|---------|---------|
| `data/tableau_idf.csv` | Matrice intensités [durée × T] |
| `data/parametres_idf.csv` | Paramètres Montana [T, a, b, R²] |
| `figures/n7/fig1_maxima_duree.png` | Maxima annuels par durée |
| `figures/n7/fig2_courbes_idf.png` | Courbes IDF (axes log-log) |
| `figures/n7/fig3_parametres_montana.png` | Évolution de a(T) et b(T) |
| `figures/n7/fig4_idf_enveloppe.png` | Enveloppe IC 95% |

---
**Auteur :** AY2K
**Encadreur :** Prof. Titembaye Donald
**Établissement :** École Polytechnique de Lomé (EPL)
**Projet :** Modélisation des courbes IDF — Station de Kara, Nord Togo


## 0. Configuration et imports


In [ ]:
import os
import warnings

warnings.filterwarnings('ignore', category=RuntimeWarning, module='scipy')

NB      = 'n7'
FIG_DIR = f'figures/{NB}'
os.makedirs(FIG_DIR, exist_ok=True)

print(f'Répertoire figures : {FIG_DIR}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy import stats as sp_stats
from scipy.optimize import curve_fit
from scipy.interpolate import PchipInterpolator

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.6,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'figure.dpi':       100,
})

print('Imports chargés')

---
## Section 1 — Chargement des données

On charge le pluviographe nettoyé avec durées standardisées (produit par n3 corrigé).
Les durées disponibles sont : 30, 60, 90, 120, 180, 240, 360, 720 min.


In [ ]:
df_pg = pd.read_csv('data/pluviographe_clean.csv', parse_dates=['date'])

# Durées standards effectivement présentes
DUREES_STANDARDS = sorted(df_pg['duree_min'].unique())
PERIODES_RETOUR  = [2, 5, 10, 20, 50, 100]
N_MIN_ANNEES     = 5   # seuil minimum d'années pour l'ajustement statistique

print(f'Pluviographe   : {len(df_pg)} lignes')
print(f'Durées présentes : {DUREES_STANDARDS}')
print(f'Seuil minimum  : {N_MIN_ANNEES} années de maxima annuels')
print(f'Périodes de retour : {PERIODES_RETOUR} ans')
print()
df_pg.head(5)

---
## Section 2 — Maxima annuels par durée

Pour chaque durée standard, on calcule le **maximum annuel d'intensité**.
Seules les durées ayant au moins 5 années de données sont conservées
pour garantir la fiabilité statistique de l'ajustement Gumbel.


In [ ]:
df_pg['annee'] = df_pg['date'].dt.year

maxima_par_duree = {}   # duree → tableau des maxima annuels (valeurs + années)

print('=== Maxima annuels par durée ===')
for duree in DUREES_STANDARDS:
    df_d    = df_pg[(df_pg['duree_min'] == duree) & (~df_pg['flag_outlier'])]
    max_ann = df_d.groupby('annee')['intensite_mm_h'].max().dropna()
    n_ann   = len(max_ann)
    if n_ann >= N_MIN_ANNEES:
        maxima_par_duree[duree] = {
            'valeurs': max_ann.values,
            'annees':  max_ann.index.values,
        }
        print(f'  {duree:>4} min : {n_ann} années | '
              f'max={max_ann.max():.1f} mm/h | moy={max_ann.mean():.1f} mm/h  ✓')
    else:
        print(f'  {duree:>4} min : {n_ann} années < {N_MIN_ANNEES} → IGNORÉE')

print(f'\n{len(maxima_par_duree)} durées retenues sur {len(DUREES_STANDARDS)}')
DUREES_RETENUES = sorted(maxima_par_duree.keys())

---
## Section 3 — Ajustement Gumbel par durée et construction du tableau IDF

Pour chaque durée retenue, on ajuste une **loi de Gumbel** sur les maxima annuels
(méthode des moments ou maximum de vraisemblance) et on calcule les valeurs de
retour pour chaque période standard.

La loi de Gumbel est recommandée par l'OMM pour les maxima annuels en hydrologie.


In [ ]:
# ==============================================================================
# AJUSTEMENT GUMBEL PAR DURÉE
# ==============================================================================
idf_table = {}   # idf_table[duree][T] = intensité (mm/h)

print('=== Ajustement Gumbel par durée ===')
for duree in DUREES_RETENUES:
    vals = maxima_par_duree[duree]['valeurs']
    try:
        params = sp_stats.gumbel_r.fit(vals)
        idf_table[duree] = {}
        for T in PERIODES_RETOUR:
            p = 1 - 1 / T
            idf_table[duree][T] = sp_stats.gumbel_r.ppf(p, *params)
        print(f'  {duree:>4} min : loc={params[0]:.2f}, scale={params[1]:.2f}  ✓')
    except Exception as e:
        # Fallback : quantiles empiriques si Gumbel ne converge pas
        print(f'  {duree:>4} min : Gumbel non convergée ({e}) → quantiles empiriques')
        idf_table[duree] = {}
        for T in PERIODES_RETOUR:
            p = min(1 - 1 / T, 1 - 1 / len(vals))
            idf_table[duree][T] = np.quantile(vals, p)

# Tableau IDF brut
df_idf_brut = pd.DataFrame(idf_table).T
df_idf_brut.index.name = 'duree_min'
print('\n=== Tableau IDF brut (avant correction monotonie) ===')
print(df_idf_brut.round(1).to_string())

---
## Section 4 — Vérification et correction de la monotonie

### Propriété fondamentale des courbes IDF

Pour une période de retour donnée, l'intensité doit **décroître strictement**
avec la durée : $I(D_1) > I(D_2)$ pour $D_1 < D_2$.

Cette propriété est une contrainte physique : une intensité moyenne sur 2 heures
ne peut pas être supérieure à une intensité moyenne sur 1 heure pour un même événement.

### Méthode de correction — Interpolation PCHIP monotone

Si des violations de monotonie sont détectées, on applique une interpolation
**PCHIP** (Piecewise Cubic Hermite Interpolating Polynomial) qui garantit la monotonie
tout en préservant la forme générale de la courbe.

On recalcule les valeurs IDF sur les durées standard en partant des durées
qui respectent déjà la monotonie (en les sélectionnant par tri décroissant).


In [ ]:
def verifier_monotonie(idf_dict, periodes):
    """
    Vérifie que les intensités décroissent avec la durée.
    Retourne le nombre de violations pour chaque période de retour.
    """
    durees = sorted(idf_dict.keys())
    violations = {}
    for T in periodes:
        n_viol = 0
        for i in range(len(durees) - 1):
            d1, d2 = durees[i], durees[i + 1]
            if d1 in idf_dict and d2 in idf_dict:
                if idf_dict[d1][T] <= idf_dict[d2][T]:
                    n_viol += 1
        violations[T] = n_viol
    return violations


def corriger_monotonie_pchip(idf_dict, periodes):
    """
    Corrige les violations de monotonie par interpolation PCHIP monotone.
    
    Stratégie : pour chaque période de retour, on trie les (durée, intensité)
    en s'assurant que la séquence est décroissante. Les points non-monotones
    sont remplacés par interpolation.
    """
    durees = sorted(idf_dict.keys())
    idf_corrige = {d: {} for d in durees}

    for T in periodes:
        vals = np.array([idf_dict[d][T] for d in durees])
        durees_arr = np.array(durees, dtype=float)

        # Extraire les points qui forment déjà une enveloppe décroissante
        # (algorithme "pool adjacent violators" simplifié)
        idx_mono = [0]
        for i in range(1, len(vals)):
            if vals[i] < vals[idx_mono[-1]]:
                idx_mono.append(i)

        d_mono = durees_arr[idx_mono]
        v_mono = vals[idx_mono]

        if len(d_mono) >= 2:
            # Interpolation PCHIP sur les durées corrigées
            # On travaille en log-log pour préserver la structure puissance
            log_d = np.log(d_mono)
            log_v = np.log(v_mono)
            interp = PchipInterpolator(log_d, log_v, extrapolate=True)
            for d in durees:
                idf_corrige[d][T] = float(np.exp(interp(np.log(d))))
        else:
            # Pas assez de points monotones → on garde les valeurs originales
            for d in durees:
                idf_corrige[d][T] = idf_dict[d][T]

    return idf_corrige


# --- Diagnostic avant correction ---
print('=== Diagnostic de monotonie (tableau brut) ===')
violations_avant = verifier_monotonie(idf_table, PERIODES_RETOUR)
total_viol = sum(violations_avant.values())
n_intervalles = len(DUREES_RETENUES) - 1
print(f'Violations totales : {total_viol} sur {n_intervalles * len(PERIODES_RETOUR)} intervalles')
for T, n_v in violations_avant.items():
    statut = '✓' if n_v == 0 else f'⚠ {n_v} violations'
    print(f'  T={T:>3} ans : {statut}')

# --- Correction si nécessaire ---
if total_viol > 0:
    print(f'\n→ Correction PCHIP appliquée ({total_viol} violations)')
    idf_table_final = corriger_monotonie_pchip(idf_table, PERIODES_RETOUR)
    
    # Vérifier après correction
    violations_apres = verifier_monotonie(idf_table_final, PERIODES_RETOUR)
    total_apres = sum(violations_apres.values())
    print(f'→ Violations après correction : {total_apres}')
else:
    print('\n→ Tableau monotone : aucune correction nécessaire')
    idf_table_final = idf_table

In [ ]:
# --- Tableau IDF final ---
df_idf = pd.DataFrame(idf_table_final).T.sort_index()
df_idf.index.name = 'duree_min'

print('=== Tableau IDF final — Intensités (mm/h) ===')
print('     [durée en minutes, intensités en mm/h]')
print()
print(df_idf.round(1).to_string())
print()

# Vérification finale de monotonie
print('=== Vérification finale ===')
durees_f = sorted(idf_table_final.keys())
for T in PERIODES_RETOUR:
    vals_f = [idf_table_final[d][T] for d in durees_f]
    ok = all(vals_f[i] > vals_f[i+1] for i in range(len(vals_f)-1))
    print(f'  T={T:>3} ans : {"✓ monotone" if ok else "⚠ non monotone"}')

---
## Section 5 — Ajustement Montana ( I = a × D^b )

Pour chaque période de retour, on ajuste la **formule de Montana** par
moindres carrés non linéaires (`scipy.optimize.curve_fit`).

- **a** : coefficient d'échelle (croît avec T)
- **b** : exposant de durée (négatif, typiquement -0.4 à -0.7)

Si `curve_fit` ne converge pas, on utilise la régression linéaire en log-log.


In [ ]:
def montana(D, a, b):
    """Formule de Montana : I = a * D^b."""
    return a * np.power(D, b)


durees_arr       = np.array(sorted(idf_table_final.keys()), dtype=float)
parametres_montana = []

print('=== Ajustement Montana par période de retour ===')
print('Formule : I(D,T) = a(T) × D^b(T)   [I en mm/h, D en min]')
print()

for T in PERIODES_RETOUR:
    intensites = np.array([idf_table_final[d][T] for d in durees_arr])

    if len(durees_arr) < 3:
        print(f'T={T:>3} ans : insuffisant ({len(durees_arr)} points)')
        continue

    try:
        popt, pcov = curve_fit(
            montana, durees_arr, intensites,
            p0=[100.0, -0.5], maxfev=5000,
            bounds=([0.1, -2.0], [10000.0, -0.1])
        )
        a_fit, b_fit = popt
        I_pred  = montana(durees_arr, a_fit, b_fit)
        ss_res  = np.sum((intensites - I_pred) ** 2)
        ss_tot  = np.sum((intensites - intensites.mean()) ** 2)
        r2      = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        parametres_montana.append(
            {'T': T, 'a': round(a_fit, 3), 'b': round(b_fit, 4), 'R²': round(r2, 4)})
        print(f'T={T:>3} ans : a={a_fit:8.3f}, b={b_fit:.4f}, R²={r2:.4f}  ✓')
    except Exception as e:
        # Fallback : régression log-log
        try:
            slope, intercept, r_val, _, _ = sp_stats.linregress(
                np.log(durees_arr), np.log(intensites))
            a_fit = np.exp(intercept)
            b_fit = slope
            parametres_montana.append(
                {'T': T, 'a': round(a_fit, 3), 'b': round(b_fit, 4),
                 'R²': round(r_val ** 2, 4)})
            print(f'T={T:>3} ans : a={a_fit:8.3f}, b={b_fit:.4f}  (fallback log-log)')
        except Exception as e2:
            print(f'T={T:>3} ans : ÉCHEC ({e2})')

df_montana = pd.DataFrame(parametres_montana)
print()
print(df_montana.to_string(index=False))

---
## Section 6 — Figures


In [ ]:
# ==============================================================================
# FIGURE 1 — Maxima annuels par durée (vue d'ensemble)
# ==============================================================================
n_durees  = len(DUREES_RETENUES)
n_cols    = min(4, n_durees)
n_rows    = (n_durees + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
axes_flat = axes.flat if n_rows * n_cols > 1 else [axes]

cpal = plt.cm.Blues(np.linspace(0.45, 0.9, n_durees))

for ax, duree, col in zip(axes_flat, DUREES_RETENUES, cpal):
    vals   = maxima_par_duree[duree]['valeurs']
    annees = maxima_par_duree[duree]['annees']
    ax.bar(annees, vals, color=col, edgecolor='white')
    ax.set_title(f'{duree} min', fontweight='bold', fontsize=10)
    ax.set_xlabel('Année', fontsize=8)
    ax.set_ylabel('I max (mm/h)', fontsize=8)
    ax.tick_params(axis='x', rotation=45, labelsize=7)

# Masquer les axes vides
for ax in list(axes_flat)[n_durees:]:
    ax.set_visible(False)

plt.suptitle('Maxima annuels d\'intensité par durée standardisée — Kara',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig1_maxima_duree.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig1 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 2 — Courbes IDF (axes log-log) — FIGURE PRINCIPALE
# ==============================================================================
colors_T = cm.plasma(np.linspace(0.05, 0.90, len(PERIODES_RETOUR)))
D_plot   = np.logspace(np.log10(20), np.log10(800), 300)

fig, ax = plt.subplots(figsize=(12, 7))

for i, row in df_montana.iterrows():
    T       = int(row['T'])
    I_curve = montana(D_plot, row['a'], row['b'])
    ax.plot(D_plot, I_curve, '-', color=colors_T[i], linewidth=2,
            label=f'T = {T} ans  (I = {row["a"]:.1f}·D^{{{row["b"]:.3f}}})')
    # Points observés (tableau IDF)
    obs_d = [d for d in DUREES_RETENUES if T in idf_table_final.get(d, {})]
    obs_I = [idf_table_final[d][T] for d in obs_d]
    ax.scatter(obs_d, obs_I, color=colors_T[i], s=50, zorder=6, marker='o')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Durée D (min)', fontsize=13)
ax.set_ylabel('Intensité I (mm/h)', fontsize=13)
ax.set_title('Courbes IDF — Station de Kara, Togo\nMéthode Montana · Loi de Gumbel',
             fontsize=14, fontweight='bold')
ax.legend(title='Période de retour', fontsize=9, title_fontsize=10, loc='upper right')
ax.grid(True, which='both', alpha=0.3)

# Annotations des durées
for d in DUREES_RETENUES:
    ax.axvline(d, color='grey', linestyle=':', alpha=0.3, linewidth=0.8)

plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig2_courbes_idf.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig2 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 3 — Évolution des paramètres a et b de Montana
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Paramètre a
ax1 = axes[0]
ax1.plot(df_montana['T'], df_montana['a'], 'o-', color='#1565C0',
         linewidth=2, markersize=8)
for _, row in df_montana.iterrows():
    ax1.annotate(f'{row["a"]:.1f}', (row['T'], row['a']),
                 textcoords='offset points', xytext=(5, 5), fontsize=9)
ax1.set_xlabel('Période de retour T (ans)')
ax1.set_ylabel('Paramètre a')
ax1.set_title('Paramètre a(T) — coefficient d\'échelle', fontweight='bold')
ax1.set_xscale('log')
ax1.set_ylim(bottom=0)

# Paramètre b
ax2 = axes[1]
ax2.plot(df_montana['T'], df_montana['b'], 's--', color='#C62828',
         linewidth=2, markersize=8)
for _, row in df_montana.iterrows():
    ax2.annotate(f'{row["b"]:.3f}', (row['T'], row['b']),
                 textcoords='offset points', xytext=(5, 3), fontsize=9)
ax2.axhspan(-0.7, -0.4, color='#E8F5E9', alpha=0.5,
            label='Plage typique (-0.4 à -0.7)')
ax2.set_xlabel('Période de retour T (ans)')
ax2.set_ylabel('Paramètre b')
ax2.set_title('Paramètre b(T) — exposant de durée', fontweight='bold')
ax2.set_xscale('log')
ax2.legend(fontsize=9)

plt.suptitle('Paramètres Montana — Station de Kara', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig3_parametres_montana.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig3 sauvegardée')

In [ ]:
# ==============================================================================
# FIGURE 4 — Enveloppe IC 95% (Bootstrap) pour T=10 et T=100 ans
# ==============================================================================
T_ref_list = [T for T in [10, 100] if T in df_montana['T'].values]
N_BOOT     = 500
np.random.seed(42)

fig, ax = plt.subplots(figsize=(12, 7))
colors_env = {10: '#1565C0', 100: '#B71C1C'}

for T_ref in T_ref_list:
    row_mont = df_montana[df_montana['T'] == T_ref]
    if row_mont.empty:
        continue
    a_ref = row_mont.iloc[0]['a']
    b_ref = row_mont.iloc[0]['b']
    col   = colors_env.get(T_ref, 'steelblue')

    # Courbe principale
    I_main = montana(D_plot, a_ref, b_ref)
    ax.plot(D_plot, I_main, '-', color=col, linewidth=2.5, label=f'T={T_ref} ans')

    # Bootstrap
    I_boot = np.full((N_BOOT, len(D_plot)), np.nan)
    for b_i in range(N_BOOT):
        sampled_d, sampled_I = [], []
        for d in DUREES_RETENUES:
            if T_ref in idf_table_final.get(d, {}):
                max_d = maxima_par_duree.get(d, {}).get('valeurs')
                if max_d is not None and len(max_d) > 0:
                    sample = np.random.choice(max_d, size=len(max_d), replace=True)
                    try:
                        p_gum  = sp_stats.gumbel_r.fit(sample)
                        i_boot = sp_stats.gumbel_r.ppf(1 - 1 / T_ref, *p_gum)
                        sampled_d.append(d)
                        sampled_I.append(i_boot)
                    except Exception:
                        pass
        if len(sampled_d) >= 3:
            try:
                popt_b, _ = curve_fit(
                    montana, sampled_d, sampled_I,
                    p0=[a_ref, b_ref], maxfev=2000,
                    bounds=([0.1, -2.0], [10000.0, -0.1]))
                I_boot[b_i] = montana(D_plot, *popt_b)
            except Exception:
                I_boot[b_i] = I_main

    I_lo = np.nanpercentile(I_boot, 2.5, axis=0)
    I_hi = np.nanpercentile(I_boot, 97.5, axis=0)
    ax.fill_between(D_plot, I_lo, I_hi, alpha=0.18, color=col,
                    label=f'IC 95% T={T_ref}')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Durée D (min)', fontsize=13)
ax.set_ylabel('Intensité I (mm/h)', fontsize=13)
ax.set_title('Courbes IDF avec enveloppe IC 95% — Station Kara',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/fig4_idf_enveloppe.png', dpi=150, bbox_inches='tight')
plt.show()
print('fig4 sauvegardée')

---
## Section 7 — Export des résultats


In [ ]:
# --- Tableau IDF ---
df_idf_export = df_idf.copy()
df_idf_export.to_csv('data/tableau_idf.csv')
print(f'data/tableau_idf.csv exporté ({len(df_idf_export)} durées)')

# --- Paramètres Montana ---
df_montana.to_csv('data/parametres_idf.csv', index=False)
print(f'data/parametres_idf.csv exporté ({len(df_montana)} périodes)')

print()
print('=== TABLEAU IDF FINAL (mm/h) ===')
print(df_idf.round(1).to_string())
print()
print('=== PARAMÈTRES MONTANA FINAUX ===')
print('Formule : I(D,T) = a(T) × D^b(T)   [I en mm/h, D en min]')
print(df_montana.to_string(index=False))

---
## Résumé final du pipeline

| Notebook | Entrée | Sortie |
|----------|--------|--------|
| n1 | `KARA PLUIE JOURNALIERE.xls` | Figures d'exploration |
| n2 | `INTENSITES DE PLUIE -KARA.xlsx` | Figures d'exploration |
| n3 | Fichiers bruts | `pluviometre_clean.csv`, `pluviographe_clean.csv` (durées standardisées) |
| n4 | CSV nettoyés | `tableau_entrainement.csv` |
| n5 | `tableau_entrainement.csv` | `serie_complete.csv` |
| n6 | `serie_complete.csv` | Valeurs de retour, tests d'adéquation |
| **n7** | `pluviographe_clean.csv` | **`tableau_idf.csv`, `parametres_idf.csv`** |

### Formule Montana retenue

$$I(D, T) = a(T) \times D^{b(T)}$$

où $I$ en mm/h, $D$ en minutes. Les paramètres sont dans `data/parametres_idf.csv`.

### Points de validation effectués

- Durées standardisées (8 durées : 30, 60, 90, 120, 180, 240, 360, 720 min)
- Seuil : minimum 5 années de maxima annuels par durée
- Monotonie vérifiée et corrigée par PCHIP si nécessaire
- IC 95% par bootstrap (n=500)

---
*Fin du Notebook 7 — Pipeline IDF complet*
